# Jour 1b — Modèles de référence : k-mer + ML classique, et one-hot + CNN

Deux façons de transformer une chaîne d'ADN en quelque chose qu'un modèle peut utiliser,
avant même de toucher à un modèle de langage pré-entraîné :

1. **Fréquences de k-mers** : compter la fréquence de chaque sous-chaîne de longueur k
   (par ex. les 256 4-mers possibles), normaliser -> un vecteur de taille fixe -> ML
   classique (régression logistique).
2. **One-hot + CNN** : encoder chaque nucléotide en un vecteur one-hot de dimension 4
   (A/C/G/T), empiler en un tenseur (4, 200), et laisser un petit CNN 1D apprendre
   lui-même ses propres caractéristiques directement à partir de la séquence brute.

In [ ]:
import sys
sys.path.append("src")

import numpy as np
import torch
from data import load_all
from featurize import kmer_matrix, one_hot_batch
from models.baselines import make_kmer_classifier, OneHotCNN
from eval import evaluate_sklearn, evaluate_logits, count_params, measure_latency_sklearn, measure_latency_torch

splits = load_all("../../../data/supervised/processed")
train, val = splits["train"], splits["val"]

## Modèle de référence 1 : fréquences de k-mers + régression logistique

**À faire :** complétez le code ci-dessous pour calculer les matrices de k-mers (k=4) sur
train/val, entraîner un classifieur logistique (`make_kmer_classifier("logreg")`), puis
l'évaluer sur validation avec `evaluate_sklearn`.

In [ ]:
K = 4
X_train_kmer = kmer_matrix(train["sequence"].tolist(), k=K)
X_val_kmer = kmer_matrix(val["sequence"].tolist(), k=K)
y_train, y_val = train["label"].to_numpy(), val["label"].to_numpy()

kmer_clf = make_kmer_classifier("logreg")
kmer_clf.fit(X_train_kmer, y_train)

kmer_metrics = evaluate_sklearn(kmer_clf, X_val_kmer, y_val)
print("k-mer + logreg:", kmer_metrics)

## Modèle de référence 2 : nucléotides one-hot + petit CNN

**À faire :** encodez les fenêtres en one-hot (`one_hot_batch`, longueur 200), instanciez
`OneHotCNN`, puis entraînez-le pendant 5 époques (Adam, lr=1e-3,
`binary_cross_entropy_with_logits`, mini-lots de 256).

In [ ]:
WINDOW = 200
X_train_oh = torch.tensor(one_hot_batch(train["sequence"].tolist(), length=WINDOW))
X_val_oh = torch.tensor(one_hot_batch(val["sequence"].tolist(), length=WINDOW))
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

cnn = OneHotCNN(seq_len=WINDOW)
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3)

n = X_train_oh.shape[0]
for epoch in range(5):
    perm = torch.randperm(n)
    epoch_loss = 0.0
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        logits = cnn(X_train_oh[idx])
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, y_train_t[idx])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    print(f"epoch {epoch+1}: loss={epoch_loss/n:.4f}")

In [ ]:
with torch.no_grad():
    val_logits = cnn(X_val_oh).numpy()
cnn_metrics = evaluate_logits(val_logits, y_val)
print("one-hot CNN:", cnn_metrics)

## Comparaison

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {"model": "kmer+logreg", **kmer_metrics,
     "params": X_train_kmer.shape[1] + 1,
     "latency_ms": measure_latency_sklearn(kmer_clf, X_val_kmer[:1])},
    {"model": "onehot+CNN", **cnn_metrics,
     "params": count_params(cnn),
     "latency_ms": measure_latency_torch(cnn, X_val_oh[:1])},
])
comparison

### Point de contrôle

Vous devriez avoir un tableau accuracy/F1/params/latence pour les deux modèles de
référence. Gardez ces chiffres (ou le DataFrame `comparison`) — ils serviront pour le
graphique d'efficacité du Jour 4.

Suite : `02_evo2_embeddings_and_classifier.ipynb` — un modèle de fondation génomique
fait-il mieux ?